# Step 6: Training 2-Stage Neural Network (MLP & LSTM)

Model 2-stage menggunakan arsitektur **Multi-Layer Perceptron (MLP)** dan **Long Short-Term Memory (LSTM)**:

| Stage | Input | Output | Model |
|-------|-------|--------|-------|
| Stage 1 | 98 fitur | Suit (C/D/H/S/N) | MLP atau LSTM |
| Stage 2 | 98 fitur | Kategori (partscore/game/small_slam/grand_slam) | MLP atau LSTM |

Level final = konversi kategori+suit ke level minimum yang valid.

**Tersedia dua pilihan model:**
- **MLP (Multi-Layer Perceptron)**: Lebih cocok untuk tabular features. Arsitektur: Input → Dense(256) → Dense(128) → Dense(64) → Output
- **LSTM (Long Short-Term Memory)**: Untuk temporal/sequence features. Features direshape menjadi sequence.

**Prasyarat:** Jalankan `05_dataset.ipynb` terlebih dahulu.

In [4]:
import sys
import os

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

DATA_PROCESSED = Path(ROOT) / "data" / "processed"
RESULTS        = Path(ROOT) / "results"

DATASET_CSV  = DATA_PROCESSED / "bridge_dataset.csv"
SPLIT_PATH   = RESULTS / "metrics" / "dataset_split.pkl"
MODEL_PATH   = RESULTS / "metrics" / "nn_model.h5"

print(f"Root proyek : {ROOT}")
print("Setup selesai.")

Root proyek : d:\SkripsiBBO
Setup selesai.


## 6.1 Load Dataset Split

In [5]:
from features import FEATURE_COLS
from model import prepare_features
from sklearn.model_selection import train_test_split

if SPLIT_PATH.exists():
    split_data = joblib.load(SPLIT_PATH)
    X_train      = split_data["X_train"]
    X_test       = split_data["X_test"]
    y_suit_train = split_data["y_suit_train"]
    y_suit_test  = split_data["y_suit_test"]
    y_cat_train  = split_data["y_cat_train"]
    y_cat_test   = split_data["y_cat_test"]
    print(f"Split data dimuat dari: {SPLIT_PATH}")
else:
    # Rebuild split dari dataset CSV
    print(f"Split file tidak ditemukan, rebuild dari {DATASET_CSV}")
    df = pd.read_csv(DATASET_CSV)
    df = df.dropna(subset=["best_contract_strain", "best_contract_category"])
    available_feat = [c for c in FEATURE_COLS if c in df.columns]
    X      = prepare_features(df, available_feat)
    y_suit = df["best_contract_strain"]
    y_cat  = df["best_contract_category"]
    X_train, X_test, y_suit_train, y_suit_test = train_test_split(
        X, y_suit, test_size=0.2, stratify=y_suit, random_state=42
    )
    _, _, y_cat_train, y_cat_test = train_test_split(
        X, y_cat, test_size=0.2, stratify=y_suit, random_state=42
    )

print(f"Train : {X_train.shape[0]} sampel")
print(f"Test  : {X_test.shape[0]} sampel")
print(f"Fitur : {X_train.shape[1]}")

Split data dimuat dari: d:\SkripsiBBO\results\metrics\dataset_split.pkl
Train : 5588 sampel
Test  : 1397 sampel
Fitur : 98


## 6.2 Cross-Validation (Opsional)

5-fold × 10 repeats sesuai C23 paper Section 5.

> **Waktu estimasi:** 10-20 menit untuk dataset besar.
> Set `RUN_CV = True` untuk menjalankan.

In [6]:
from model import train, save_model

# Pilih model: "mlp" atau "lstm"
MODEL_TYPE = "mlp"  # Ubah ke "lstm" untuk menggunakan LSTM

print(f"Training 2-Stage {MODEL_TYPE.upper()} Neural Network...")
model = train(X_train, y_suit_train, y_cat_train, model_type=MODEL_TYPE)

save_model(model, MODEL_PATH)
print(f"Model disimpan ke: {MODEL_PATH}")

Training 2-Stage MLP Neural Network...
Training 2-Stage MLP Model


AttributeError: 'NoneType' object has no attribute 'utils'

## 6.3 Training Model Utama

In [ ]:
from model import train, save_model

print("Training 2-Stage Random Forest...")
model = train(X_train, y_suit_train, y_cat_train)

save_model(model, MODEL_PATH)
print(f"Model disimpan ke: {MODEL_PATH}")

## 6.4 Sanity Check: Prediksi di Training Set

In [ ]:
pred_train = model.predict(X_train.head(10))
actual = pd.DataFrame({
    "suit_true":  y_suit_train.head(10).values,
    "cat_true":   y_cat_train.head(10).values,
})
comparison = pd.concat([actual.reset_index(drop=True), pred_train.reset_index(drop=True)], axis=1)
comparison["suit_ok"] = comparison["suit_true"] == comparison["pred_suit"]
comparison["cat_ok"]  = comparison["cat_true"]  == comparison["pred_category"]
print("Preview 10 prediksi pertama (training set):")
comparison

## 6.5 Preview Feature Importance

In [ ]:
imp_suit, imp_cat = model.feature_importance()

print("Top 10 Fitur Terpenting — Stage 1 (Suit):")
print(imp_suit.head(10).to_string())
print()
print("Top 10 Fitur Terpenting — Stage 2 (Category):")
print(imp_cat.head(10).to_string())

---
## Output

File yang dihasilkan:
- `results/metrics/rf_model.pkl` — model TwoStageRF tersimpan
- `results/metrics/cv_results.pkl` — hasil CV (jika RUN_CV=True)

**Langkah berikutnya:** Buka `07_evaluasi.ipynb` untuk evaluasi model.